# FullTextPDF Pipeline

## Goal
Find the **minimum chunk size** needed for Claude to accurately extract biochemical reactions from a PDF, by comparing extracted output against manually curated Reactome data using semantic similarity.

## Pipeline
1. **Input** — PDF of a PubMed full-text paper
2. **Auto-detect gene** — Claude reads the PDF and identifies the primary gene
3. **Chunk** — split into `[500, 1000, 2000, full_text]` word chunks
4. **Extract** — Claude extracts biochemical reactions from each chunk in `AnnotationResult` format
5. **Ground truth** — pull manually curated reactions from Neo4j (pathway summaries + reaction roles)
6. **Verify** — embed both extracted and ground truth using SentenceTransformer, compute cosine similarity
7. **Compare** — find the smallest chunk that produces high similarity to ground truth

## Cell 1 — Install Dependencies

In [9]:
# Run once if needed
# !pip install pymupdf anthropic python-dotenv pandas sentence-transformers scikit-learn neo4j

## Cell 2 — Imports & Setup

In [10]:
import os
import sys
import glob
import json
import time
import numpy as np
import pandas as pd
import fitz  # PyMuPDF
import anthropic
from dotenv import load_dotenv

# ── Load .env FIRST with explicit path before any other imports that need env vars
PROJECT_ROOT = os.path.expanduser('~/curator-tool-llm')
dotenv_path = os.path.join(PROJECT_ROOT, '.env')
load_dotenv(dotenv_path=dotenv_path, override=True)

# ── Force vars into os.environ so ReactomeNeo4jUtils picks them up at import time
required_vars = ['REACTOME_NEO4J_URI', 'REACTOME_NEO4J_USER', 'REACTOME_NEO4J_PWD', 'REACTOME_NEO4J_DATABASE']
for var in required_vars:
    val = os.getenv(var)
    if val:
        os.environ[var] = val
    else:
        print(f'❌ Missing env var: {var} — check {dotenv_path}')

# ── Now safe to import ReactomeNeo4jUtils
REACTOME_LLM_PATH = os.path.join(PROJECT_ROOT, 'reactome_llm')
if REACTOME_LLM_PATH not in sys.path:
    sys.path.insert(0, REACTOME_LLM_PATH)
import ReactomeNeo4jUtils as neo4jutils

# ── Anthropic client
api_key = os.getenv('ANTHROPIC_API_KEY')
if not api_key:
    print('❌ ANTHROPIC_API_KEY not found — check .env')
else:
    client = anthropic.Anthropic(api_key=api_key)
    print('✅ Anthropic client initialized')

MODEL_NAME = 'claude-sonnet-4-6'

# ── SentenceTransformer — same model used in GenePathwayAnnotator.py
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
EMBED_MODEL_NAME = 'pritamdeka/S-PubMedBert-MS-MARCO'
print(f'Loading embedding model: {EMBED_MODEL_NAME} (cached after first run)...')
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print('✅ Embedding model loaded')
print(f'✅ Neo4j URI: {os.getenv("REACTOME_NEO4J_URI")}')
print(f'✅ Neo4j DB:  {os.getenv("REACTOME_NEO4J_DATABASE")}')
print(f'✅ Model:     {MODEL_NAME}')


✅ Anthropic client initialized
Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO (cached after first run)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Embedding model loaded
✅ Neo4j URI: bolt://localhost:7687
✅ Neo4j DB:  graph
✅ Model:     claude-sonnet-4-6


## Cell 3 — Define Inputs

In [11]:
# Drop any PDFs into data/papers/ — no renaming needed
PDF_FOLDER = os.path.join(PROJECT_ROOT, 'data', 'papers')
os.makedirs(PDF_FOLDER, exist_ok=True)
PDF_PATHS = glob.glob(os.path.join(PDF_FOLDER, '*.pdf'))

# Chunk sizes in words. None = full PDF with no chunking.
# We test progressively larger chunks to find the minimum needed.
CHUNK_SIZES = [500, 1000, 2000, None]

if not PDF_PATHS:
    print(f'❌ No PDFs found in {PDF_FOLDER}')
    print('  → Copy your PDF files there, then re-run this cell')
else:
    print(f'Found {len(PDF_PATHS)} PDF(s):')
    for p in PDF_PATHS:
        print(f'  - {os.path.basename(p)}')
print(f'Chunk sizes to test: {CHUNK_SIZES}')

Found 2 PDF(s):
  - ZNFX1.pdf
  - PINK1.pdf
Chunk sizes to test: [500, 1000, 2000, None]


## Cell 4 — Extract PDF Text & Auto-Detect Gene

In [12]:
def extract_text_from_pdf(pdf_path):
    """Extract full text from a PDF using PyMuPDF."""
    if not os.path.exists(pdf_path):
        print(f'❌ File not found: {pdf_path}')
        return None, 0
    doc = fitz.open(pdf_path)
    page_count = doc.page_count
    full_text = ''
    for page in doc:
        full_text += page.get_text()
    doc.close()
    print(f'✅ Extracted {len(full_text.split()):,} words from {os.path.basename(pdf_path)} ({page_count} pages)')
    return full_text, page_count


def autodetect_gene(text, filename=''):
    """
    Smarter gene detection:
    - Reads the first page (title + abstract) which is most informative
    - Also reads the last 500 words (conclusions) for confirmation
    - Asks Claude to reason about which gene is the PRIMARY focus,
      not just the most mentioned (e.g. ubiquitin is mentioned a lot
      in PINK1 papers but PINK1 is the primary gene)
    - Uses filename as a hint if available
    """
    words = text.split()
    # First 1000 words = title + abstract (most reliable signal)
    opening = ' '.join(words[:1000])
    # Last 300 words = conclusion (confirms the main finding)
    closing = ' '.join(words[-300:])
    # Filename hint (e.g. PINK1.pdf is a strong signal)
    fname_hint = f'The filename is "{filename}" which may hint at the gene.' if filename else ''

    prompt = f"""You are identifying the PRIMARY gene that a scientific paper is studying.

{fname_hint}

Rules:
- The primary gene is the one being STUDIED or CHARACTERIZED, not just mentioned
- Ubiquitous proteins (ubiquitin, actin, tubulin, GAPDH) are almost never the primary gene
- If the filename contains a gene name, that is a very strong hint
- Return the official HGNC gene symbol (e.g. PINK1, not Pink1 or PINK-1)

Paper opening (title + abstract):
{opening}

Paper closing (conclusions):
{closing}

Return ONLY a JSON object, no markdown:
{{"gene": "<HGNC symbol>", "full_name": "<full protein name>", "confidence": <float 0-1>, "reasoning": "<one sentence why>"}}"""

    try:
        message = client.messages.create(
            model=MODEL_NAME,
            max_tokens=300,
            temperature=0,
            messages=[{'role': 'user', 'content': prompt}]
        )
        result = message.content[0].text.strip().replace('```json','').replace('```','').strip()
        return json.loads(result)
    except Exception as e:
        print(f'  ⚠️ Gene detection failed: {e}')
        return {'gene': 'UNKNOWN', 'full_name': 'unknown', 'confidence': 0, 'reasoning': 'detection failed'}


# Process all PDFs
pdf_texts = {}
pdf_genes = {}

for path in PDF_PATHS:
    text, pages = extract_text_from_pdf(path)
    if text:
        fname = os.path.basename(path)
        pdf_texts[fname] = text
        print(f'  Detecting primary gene from {fname}...', end=' ')
        gene_info = autodetect_gene(text, filename=fname)
        pdf_genes[fname] = gene_info
        print(f"→ {gene_info['gene']} ({gene_info['full_name']})")
        print(f"    confidence={gene_info['confidence']:.0%} | {gene_info['reasoning']}")

print(f'\n✅ Processed {len(pdf_texts)} PDF(s)')
print('\n⚠️  If any gene looks wrong, manually override it like this:')
print('    pdf_genes["PINK1.pdf"]["gene"] = "PINK1"')


✅ Extracted 19,064 words from ZNFX1.pdf (35 pages)
  Detecting primary gene from ZNFX1.pdf... → ZNFX1 (zinc finger NFX1-type containing 1)
    confidence=99% | The paper is entirely focused on characterizing the structure, mechanism, and function of ZNFX1 as a non-canonical E3 ubiquitin ligase involved in innate immunity, as confirmed by the filename, title, abstract, and experimental data throughout.
✅ Extracted 7,858 words from PINK1.pdf (11 pages)
  Detecting primary gene from PINK1.pdf... → PINK1 (PTEN induced kinase 1)
    confidence=95% | The paper directly studies PINK1 kinase activity, characterizing its phosphorylation of ubiquitin at serine 65 as a mechanism for activating Parkin, with the filename and throughout the text confirming PINK1 as the primary subject of investigation.

✅ Processed 2 PDF(s)

⚠️  If any gene looks wrong, manually override it like this:
    pdf_genes["PINK1.pdf"]["gene"] = "PINK1"


## Cell 5 — Chunk Text

In [13]:
def chunk_text(text, chunk_size, overlap=100):
    """
    Split text into chunks of chunk_size words with overlap.
    chunk_size=None returns the entire text as one chunk (full PDF).
    """
    if chunk_size is None:
        return [text]
    words = text.split()
    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks


# Build chunk dict: {filename: {chunk_label: [chunks]}}
all_chunks = {}
for fname, text in pdf_texts.items():
    all_chunks[fname] = {}
    for size in CHUNK_SIZES:
        label = 'full_text' if size is None else size
        chunks = chunk_text(text, chunk_size=size)
        all_chunks[fname][label] = chunks
        desc = 'full PDF (no chunking)' if size is None else f'{size}-word chunks'
        print(f'{fname} | {desc}: {len(chunks)} chunk(s)')

print('\n✅ Chunking complete')

ZNFX1.pdf | 500-word chunks: 48 chunk(s)
ZNFX1.pdf | 1000-word chunks: 22 chunk(s)
ZNFX1.pdf | 2000-word chunks: 11 chunk(s)
ZNFX1.pdf | full PDF (no chunking): 1 chunk(s)
PINK1.pdf | 500-word chunks: 20 chunk(s)
PINK1.pdf | 1000-word chunks: 9 chunk(s)
PINK1.pdf | 2000-word chunks: 5 chunk(s)
PINK1.pdf | full PDF (no chunking): 1 chunk(s)

✅ Chunking complete


## Cell 6 — Extract the Single Main Biochemical Reaction
For each chunk, Claude identifies the **one most biochemically significant reaction** the gene participates in (the key finding — not a list of every reaction mentioned) and expresses it in Reactome relationship format:

```
PINK1 - phosphorylates -> ubiquitin
PINK1 - localizes_to -> mitochondrial membrane
```

Each chunk is later scored **independently** against the Neo4j ground truth (Cell 9). Full text is just one more condition — it is **not** treated as a gold standard.

In [ ]:
# Import prompts from dedicated script
import importlib.util

prompts_path = os.path.join(PROJECT_ROOT, 'reactome_llm', 'FullTextPDFPrompts.py')
spec = importlib.util.spec_from_file_location('prompts', prompts_path)
prompts_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prompts_mod)
print(f'✅ Prompts loaded from {prompts_path}')


def extract_reactions(text_chunk, gene):
    """
    Screen + extract:
      1. Extracts ALL ReactionlikeEvent instances found in each chunk
      3. Returns empty reactions list for non-reaction chunks

    Output follows Reactome ReactionlikeEvent data model:
      name, reactionType, input, output, catalystActivity,
      regulatedBy, compartment, summation, relationships

    NOTE: confidence is Claude's SELF-ASSESSED rating (subjective, 0-1).
    Objective accuracy is measured via cosine similarity vs Neo4j.
    """
    prompt = prompts_mod.build_extraction_prompt(gene, text_chunk)
    try:
        message = client.messages.create(
            model=MODEL_NAME,
            max_tokens=2048,
            temperature=0.1,
            messages=[{'role': 'user', 'content': prompt}]
        )
        result = message.content[0].text.strip().replace('```json', '').replace('```', '').strip()
        return json.loads(result)
    except Exception as e:
        print(f'    ⚠️ Extraction error: {e}')
        return None


print('✅ Extraction function defined')
print()
print('SCREENING + EXTRACTION:')
print('  all ReactionlikeEvent instances extracted')
print('OUTPUT FORMAT (per reaction, Reactome ReactionlikeEvent data model):')
print('  name, reactionType, input, output, catalystActivity,')
print('  regulatedBy, compartment, summation, relationships, confidence')
print()
print('SCORING:')
print('  confidence        = Claude self-assessed (subjective, 0-1)')
print('  sim_vs_reactions  = cosine similarity vs Neo4j reaction roles (objective)')

## Cell 7 — Run Extraction on All Chunks
For every chunk, prints the **actual main reaction** Claude extracted (and its relationship lines) — not just a count. Each chunk is stored separately for independent scoring in Cell 9.

⚠️ Keep `TEST_MODE = True` for the first run — only the first chunk per size is extracted.

In [ ]:
TEST_MODE = True  # Set False to run all chunks

extraction_results = []  # stores all results for comparison

for fname, size_dict in all_chunks.items():
    gene = pdf_genes[fname]['gene']
    print(f'\n📄 {fname} | gene: {gene}')

    for chunk_label, chunks in size_dict.items():
        chunks_to_run = chunks[:1] if TEST_MODE else chunks
        label_str = 'full text' if chunk_label == 'full_text' else f'{chunk_label}-word chunks'
        print(f'  [{label_str}] — {len(chunks_to_run)} chunk(s) to extract...')

        for i, chunk in enumerate(chunks_to_run):
            print(f'    Chunk {i+1}/{len(chunks_to_run)}...')
            result = extract_reactions(chunk, gene)
            reactions = result.get('reactions', []) if result else None

            if reactions:
                print(f'      ✅ {len(reactions)} reaction(s) extracted:')
                for r in reactions:
                    print(f"         • {r.get('name', '(no name)')}  (confidence={r.get('confidence', 0)})")
                    for rel in r.get('relationships', []):
                        print(f'             - {rel}')
            elif result is not None:
                print('      — no reactions in this chunk')
            else:
                print('      ❌ extraction failed')

            extraction_results.append({
                'source': fname,
                'gene': gene,
                'chunk_size': chunk_label,
                'chunk_index': i,
                'word_count': len(chunk.split()),
                'chunk_preview': chunk[:200] + '...',
                'extraction': result  # {"reactions": [...]}
            })
            time.sleep(0.3)

print(f'\n✅ Extraction complete — {len(extraction_results)} total results')

In [21]:
# Patching Neo4jUtils by overwriting the variables
neo4jutils.URI = "bolt://localhost:7687"
neo4jutils.AUTH = ("neo4j", os.getenv("REACTOME_NEO4J_PWD"))
neo4jutils.DB = "graph.db"

## Cell 8 — Pull Ground Truth from Neo4j
Fetches manually curated data for each detected gene:
- **Pathway summaries** — descriptive text of pathways the gene participates in
- **Reaction roles** — specific reactions and roles (reactant/catalyst/regulator)

In [22]:
def get_ground_truth(gene):
    """
    Pull manually curated ground truth from Neo4j for a gene.
    Returns two separate text strings:
      - pathway_summary_text: concatenated pathway summaries
      - reaction_roles_text: concatenated reaction + role descriptions
    """
    # 1. Get pathways for this gene
    pathways = neo4jutils.query_pathways_for_gene(gene)
    if not pathways:
        print(f'  ⚠️ No pathways found in Neo4j for gene: {gene}')
        return None, None

    print(f'  Found {len(pathways)} pathway(s) for {gene}')

    # 2. Pathway summaries
    summary_texts = []
    for p in pathways[:10]:  # limit to top 10 to avoid token overflow
        summary = neo4jutils.query_pathway_summary(p['pathway'])
        if summary:
            summary_texts.append(f"Pathway: {p['pathway']}\n{summary}")

    pathway_summary_text = '\n\n'.join(summary_texts) if summary_texts else ''

    # 3. Reaction roles — what does the gene actually do in each pathway?
    reaction_texts = []
    for p in pathways[:10]:
        try:
            roles_df = neo4jutils.query_reaction_roles_of_pathway(p['pathway'], [gene])
            if roles_df is not None and not roles_df.empty:
                for _, row in roles_df.iterrows():
                    reaction_texts.append(
                        f"{gene} acts as {row['role']} in reaction '{row['reaction']}' "
                        f"within pathway '{row['pathway']}'"
                    )
        except Exception as e:
            print(f'    ⚠️ Reaction query failed for {p["pathway"]}: {e}')

    reaction_roles_text = '\n'.join(reaction_texts) if reaction_texts else ''

    return pathway_summary_text, reaction_roles_text


# Fetch ground truth for each unique gene
ground_truth = {}  # gene -> {'pathway_summaries': str, 'reaction_roles': str}
unique_genes = list(set(info['gene'] for info in pdf_genes.values()))

for gene in unique_genes:
    print(f'\nFetching Neo4j ground truth for: {gene}')
    pathway_text, reaction_text = get_ground_truth(gene)
    ground_truth[gene] = {
        'pathway_summaries': pathway_text or '',
        'reaction_roles': reaction_text or ''
    }
    print(f'  Pathway summary text: {len((pathway_text or "").split()):,} words')
    print(f'  Reaction roles text:  {len((reaction_text or "").split()):,} words')

print('\n✅ Ground truth loaded')


Fetching Neo4j ground truth for: ZNFX1
  ⚠️ No pathways found in Neo4j for gene: ZNFX1
  Pathway summary text: 0 words
  Reaction roles text:  0 words

Fetching Neo4j ground truth for: PINK1
  Found 14 pathway(s) for PINK1


/Users/aileenzheng/miniconda3/envs/paperqa/lib/python3.10/site-packages/neo4j/_sync/work/result.py:636: UserWarning: Expected a result with a single record, but found multiple.
  warn(
Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="Binding relationships to a list in a variable length pattern is deprecated. (Binding a variable length relationship pattern to a variable ('r_role') is deprecated and will be unsupported in a future version. The recommended way is to bind the whole path to a variable, then extract the relationships:\n\tMATCH p = (...)-[...]-(...)\n\tWITH *, relationships(p) AS r_role)", position=<SummaryInputPosition line=4, column=19, offset=132>, raw_classification=None, classification=<NotificationClassification.UNKNOWN: 'UNKNOWN'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_severity': 'WARNING', '_position'

  Pathway summary text: 3,825 words
  Reaction roles text:  1,024 words

✅ Ground truth loaded


In [20]:
# Debug Cell 8 
from neo4j import GraphDatabase
import os

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", os.getenv("REACTOME_NEO4J_PWD"))
)
with driver.session(database="system") as session:
    result = session.run("SHOW DATABASES")
    for record in result:
        print(record["name"], "—", record["currentStatus"])
driver.close()

graph.db — online
system — online


## Cell 9 — Embed & Compare (SentenceTransformer cosine similarity)
Each chunk's extracted main reaction is embedded and compared — **independently** — against both Neo4j ground-truth sources (pathway summaries and reaction roles). No chunk is measured against another; Neo4j is the only ground truth.

In [ ]:
def reaction_to_text(extraction):
    """Flatten an extraction dict ({"reactions": [...]}) into a single text
    string for embedding — concatenates each reaction's name, summation text,
    and relationship lines across all reactions in the chunk."""
    if not extraction:
        return ''
    parts = []
    for r in extraction.get('reactions', []):
        if r.get('name'):
            parts.append(r['name'])
        summ = r.get('summation') or {}
        if isinstance(summ, dict) and summ.get('text'):
            parts.append(summ['text'])
        parts.extend(r.get('relationships', []))
    return ' '.join(filter(None, parts))


def compute_similarity(text_a, text_b):
    """Embed two texts and return cosine similarity (0-1)."""
    if not text_a.strip() or not text_b.strip():
        return 0.0
    emb_a = embed_model.encode([text_a])
    emb_b = embed_model.encode([text_b])
    return float(cosine_similarity(emb_a, emb_b)[0][0])


similarity_results = []

for entry in extraction_results:
    gene = entry['gene']
    extraction = entry['extraction'] or {}
    reactions = extraction.get('reactions', [])
    extracted_text = reaction_to_text(extraction)
    gt = ground_truth.get(gene, {})

    # Objective scores: semantic similarity vs Neo4j ground truth (each chunk independently)
    sim_pathway = compute_similarity(extracted_text, gt.get('pathway_summaries', ''))
    sim_reaction = compute_similarity(extracted_text, gt.get('reaction_roles', ''))

    # Aggregate per-reaction LLM confidences into one number for the chunk
    confidences = [r.get('confidence') for r in reactions
                   if isinstance(r.get('confidence'), (int, float))]
    avg_conf = round(sum(confidences) / len(confidences), 3) if confidences else None

    similarity_results.append({
        'source': entry['source'],
        'gene': gene,
        'chunk_size': entry['chunk_size'],
        'word_count': entry['word_count'],
        'num_reactions': len(reactions),
        # Names of all reactions extracted from the chunk
        'main_reaction': '; '.join(r.get('name', '') for r in reactions),
        # ── Objective semantic similarity vs Neo4j curated data
        'sim_vs_pathway_summaries': round(sim_pathway, 4),
        'sim_vs_reaction_roles': round(sim_reaction, 4),
        'avg_similarity': round((sim_pathway + sim_reaction) / 2, 4),
        # ── LLM self-assessed relevance (subjective), averaged across reactions
        'llm_confidence': avg_conf,
        'reaction_present': len(reactions) > 0,
    })

df = pd.DataFrame(similarity_results)
print(f'✅ Similarity scoring complete — {len(df)} rows')
print()
print('Columns:')
print('  num_reactions             — how many reactions Claude extracted from the chunk')
print('  sim_vs_pathway_summaries  — objective: cosine similarity vs Neo4j pathway text')
print('  sim_vs_reaction_roles     — objective: cosine similarity vs Neo4j reaction roles')
print('  avg_similarity            — objective: average of the two above')
print('  llm_confidence            — subjective: mean Claude self-rating across extracted reactions')
print('  reaction_present          — whether Claude found any reaction in the chunk')

## Cell 10 — View Results Table

In [24]:
if 'df' not in dir() or df.empty:
    print('⚠️ No results yet — run Cells 7 and 9 first')
else:
    pd.set_option('display.max_colwidth', 60)
    display_cols = [
        'source', 'gene', 'chunk_size', 'word_count',
        'main_reaction',
        # Objective semantic similarity scores (vs Neo4j ground truth)
        'sim_vs_pathway_summaries', 'sim_vs_reaction_roles', 'avg_similarity',
        # LLM self-assessed relevance
        'llm_confidence', 'reaction_present'
    ]
    display(df[[c for c in display_cols if c in df.columns]])


,source,gene,chunk_size,word_count,main_reaction,sim_vs_pathway_summaries,sim_vs_reaction_roles,avg_similarity,llm_confidence,reaction_present
0,ZNFX1.pdf,ZNFX1,500,500,"ZNFX1, activated by single-stranded RNA-induced dimeriza...",0.0000,0.0000,0.0000,0.95,True
1,ZNFX1.pdf,ZNFX1,1000,1000,ZNFX1 acts as a nucleic-acid-activated E3 ubiquitin liga...,0.0000,0.0000,0.0000,0.97,True
2,ZNFX1.pdf,ZNFX1,2000,2000,ZNFX1 acts as a nucleic-acid-activated E3 ubiquitin liga...,0.0000,0.0000,0.0000,0.97,True
3,ZNFX1.pdf,ZNFX1,full_text,19064,ZNFX1 acts as a nucleic-acid-activated E3 ubiquitin liga...,0.0000,0.0000,0.0000,0.97,True
4,PINK1.pdf,PINK1,500,500,PINK1 phosphorylates ubiquitin at serine 65 (S65) on the...,0.9232,0.9435,0.9333,0.97,True
5,PINK1.pdf,PINK1,1000,1000,PINK1 phosphorylates ubiquitin at serine 65 on the outer...,0.9167,0.9453,0.9310,0.97,True
6,PINK1.pdf,PINK1,2000,2000,PINK1 phosphorylates ubiquitin at serine 65 on the outer...,0.9167,0.9453,0.9310,0.98,True
7,PINK1.pdf,PINK1,full_text,7858,"PINK1 phosphorylates ubiquitin at serine 65, which activ...",0.9236,0.9482,0.9359,0.97,True


## Cell 11 — Find Minimum Effective Chunk Size
Key result: what is the smallest chunk that produces similarity close to full text?

In [25]:
if 'df' not in dir() or df.empty:
    print('⚠️ No results yet')
else:
    print('=== Average similarity by chunk size ===')
    summary = df.groupby('chunk_size')[[
        'sim_vs_pathway_summaries',
        'sim_vs_reaction_roles',
        'avg_similarity',
        'llm_confidence',
    ]].mean(numeric_only=True).round(4)
    print(summary)

    # Each chunk size is scored independently against the Neo4j ground truth.
    # Full text is NOT a gold standard — it is just one more condition.
    # "Minimum effective chunk size" = the SMALLEST chunk size whose average
    # similarity to Neo4j reaches >=90% of the BEST-performing condition.
    per_size = df.groupby('chunk_size')['avg_similarity'].mean()
    best_size = per_size.idxmax()
    best_sim = per_size.max()
    threshold = best_sim * 0.90

    # full_text sorts as the largest "size" so it's only chosen as a last resort.
    def _size_key(s):
        return float('inf') if s == 'full_text' else float(s)

    print(f'\nBest-performing condition: {best_size} (avg_similarity={best_sim:.4f})')
    print(f'90% threshold: {threshold:.4f}')
    print('\nPer-condition similarity vs Neo4j ground truth:')
    for s in sorted(per_size.index, key=_size_key):
        meets = '✅' if per_size[s] >= threshold else '❌'
        print(f'  {meets} {s}: avg_similarity={per_size[s]:.4f}')

    passing = [s for s in per_size.index if per_size[s] >= threshold]
    numeric_passing = [s for s in passing if s != 'full_text']
    if numeric_passing:
        best = min(numeric_passing, key=_size_key)
        print(f'\n→ Minimum effective chunk size: {best} words')
    elif 'full_text' in passing:
        print('\n→ No sub-full chunk met the threshold — full text required')
    else:
        print('\n→ No condition met the threshold')


=== Average similarity by chunk size ===
            sim_vs_pathway_summaries  sim_vs_reaction_roles  avg_similarity  \
chunk_size                                                                    
500                           0.4616                 0.4718          0.4666   
1000                          0.4584                 0.4726          0.4655   
2000                          0.4584                 0.4726          0.4655   
full_text                     0.4618                 0.4741          0.4680   

            llm_confidence  
chunk_size                  
500                  0.960  
1000                 0.970  
2000                 0.975  
full_text            0.970  

Best-performing condition: full_text (avg_similarity=0.4679)
90% threshold: 0.4212

Per-condition similarity vs Neo4j ground truth:
  ✅ 500: avg_similarity=0.4667
  ✅ 1000: avg_similarity=0.4655
  ✅ 2000: avg_similarity=0.4655
  ✅ full_text: avg_similarity=0.4679

→ Minimum effective chunk size: 500 words


## Cell 12 — Save Results

In [26]:
if df.empty:
    print('⚠️ No results to save')
else:
    results_dir = os.path.join(PROJECT_ROOT, 'results')
    os.makedirs(results_dir, exist_ok=True)

    # Save similarity summary
    csv_path = os.path.join(results_dir, 'chunk_similarity_results.csv')
    df.to_csv(csv_path, index=False)
    print(f'✅ Similarity results saved to {csv_path}')

    # Save full extraction results (includes raw AnnotationResult dicts)
    json_path = os.path.join(results_dir, 'extraction_results.json')
    with open(json_path, 'w') as f:
        json.dump(extraction_results, f, indent=2)
    print(f'✅ Full extraction results saved to {json_path}')

✅ Similarity results saved to /Users/aileenzheng/curator-tool-llm/results/chunk_similarity_results.csv
✅ Full extraction results saved to /Users/aileenzheng/curator-tool-llm/results/extraction_results.json
